# Kalshi / Polymarket Binary Arbitrage

## Strategy
Both platforms offer binary YES/NO contracts on the same events. When the same contract is priced differently across platforms, a risk-free profit exists:

- **Arb direction A**: `YES_polymarket + NO_kalshi < 1`  → buy YES on Poly, buy NO on Kalshi
- **Arb direction B**: `YES_kalshi + NO_polymarket < 1`  → buy YES on Kalshi, buy NO on Poly

Both legs together always pay out \$1. If total cost < \$1, profit is locked in.

**Expected Value formula:**
```
EV = $1.00 - (leg_a_cost + leg_b_cost)
profit_pct = EV / (leg_a_cost + leg_b_cost)
```

## Requirements
```
pip install requests python-dotenv py-clob-client kalshi-python
```

In [ ]:
# ── Installs (run once) ────────────────────────────────────────────────────────
# !pip install requests python-dotenv py-clob-client

In [ ]:
import os
import time
import logging
from dataclasses import dataclass, field
from typing import Optional

import requests
from dotenv import load_dotenv

load_dotenv()  # expects .env with KALSHI_EMAIL, KALSHI_PASSWORD, POLY_API_KEY, POLY_API_SECRET, POLY_PASSPHRASE

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

## 1 · Configuration

In [ ]:
# ── Trading parameters ─────────────────────────────────────────────────────────
MIN_PROFIT_PCT   = 0.02   # Minimum 2 % spread after fees to fire a trade
MAX_TRADE_USD    = 100    # Max dollars per arbitrage leg
DRY_RUN          = True   # Set False to place real orders
POLL_INTERVAL_S  = 15     # Seconds between scans

# Fee estimates (taker, per platform)
KALSHI_FEE_PCT   = 0.07   # 7 % of winnings
POLY_FEE_PCT     = 0.02   # ~2 % taker fee

# ── API base URLs ──────────────────────────────────────────────────────────────
KALSHI_BASE  = "https://trading-api.kalshi.com/trade-api/v2"
POLY_BASE    = "https://clob.polymarket.com"

## 2 · Kalshi Client

In [ ]:
class KalshiClient:
    """Thin wrapper around the Kalshi REST API v2."""

    def __init__(self):
        self.session = requests.Session()
        self.token: Optional[str] = None

    # ── Auth ───────────────────────────────────────────────────────────────────
    def login(self, email: str, password: str) -> None:
        resp = self.session.post(
            f"{KALSHI_BASE}/login",
            json={"email": email, "password": password},
        )
        resp.raise_for_status()
        self.token = resp.json()["token"]
        self.session.headers.update({"Authorization": f"Bearer {self.token}"})
        log.info("Kalshi: logged in")

    # ── Market data ────────────────────────────────────────────────────────────
    def get_markets(self, limit: int = 200, cursor: str = "") -> dict:
        params = {"limit": limit, "status": "open"}
        if cursor:
            params["cursor"] = cursor
        resp = self.session.get(f"{KALSHI_BASE}/markets", params=params)
        resp.raise_for_status()
        return resp.json()

    def get_orderbook(self, ticker: str) -> dict:
        resp = self.session.get(f"{KALSHI_BASE}/markets/{ticker}/orderbook")
        resp.raise_for_status()
        return resp.json()

    def best_ask(self, ticker: str, side: str = "yes") -> Optional[float]:
        """Return best ask price (0-1 scale) for 'yes' or 'no' side."""
        try:
            ob = self.get_orderbook(ticker)
            book = ob.get("orderbook", {})
            asks = book.get(f"{side}_ask", [])  # list of [price_cents, qty]
            if not asks:
                return None
            return asks[0][0] / 100  # Kalshi prices are in cents (0-99)
        except Exception as e:
            log.warning("Kalshi orderbook error %s: %s", ticker, e)
            return None

    # ── Order placement ────────────────────────────────────────────────────────
    def place_order(
        self,
        ticker: str,
        side: str,          # 'yes' or 'no'
        count: int,         # number of contracts ($0.01 each cent of cost)
        limit_price: int,   # cents (1-99)
        client_order_id: str = "",
    ) -> dict:
        if DRY_RUN:
            log.info("[DRY RUN] Kalshi order: %s %s %d @ %d¢", ticker, side, count, limit_price)
            return {"dry_run": True}
        payload = {
            "ticker": ticker,
            "side": side,
            "count": count,
            "type": "limit",
            "yes_price" if side == "yes" else "no_price": limit_price,
            "action": "buy",
        }
        if client_order_id:
            payload["client_order_id"] = client_order_id
        resp = self.session.post(f"{KALSHI_BASE}/portfolio/orders", json=payload)
        resp.raise_for_status()
        return resp.json()

## 3 · Polymarket CLOB Client

In [ ]:
class PolymarketClient:
    """Thin wrapper around the Polymarket CLOB API."""

    def __init__(self, api_key: str, api_secret: str, passphrase: str):
        self.session = requests.Session()
        self.session.headers.update({
            "POLY-API-KEY":        api_key,
            "POLY-API-SECRET":     api_secret,
            "POLY-API-PASSPHRASE": passphrase,
        })

    # ── Market data ────────────────────────────────────────────────────────────
    def get_markets(self, next_cursor: str = "") -> dict:
        params = {"active": "true", "closed": "false"}
        if next_cursor:
            params["next_cursor"] = next_cursor
        resp = self.session.get(f"{POLY_BASE}/markets", params=params)
        resp.raise_for_status()
        return resp.json()

    def get_orderbook(self, token_id: str) -> dict:
        resp = self.session.get(f"{POLY_BASE}/book", params={"token_id": token_id})
        resp.raise_for_status()
        return resp.json()

    def best_ask(self, token_id: str) -> Optional[float]:
        """Return best ask price (0-1) for a YES token."""
        try:
            ob = self.get_orderbook(token_id)
            asks = ob.get("asks", [])  # [{price, size}, ...] sorted ascending
            if not asks:
                return None
            return float(asks[0]["price"])
        except Exception as e:
            log.warning("Poly orderbook error %s: %s", token_id, e)
            return None

    # ── Order placement ────────────────────────────────────────────────────────
    def place_order(
        self,
        token_id: str,
        side: str,       # 'BUY' or 'SELL'
        price: float,    # 0-1
        size: float,     # USDC amount
    ) -> dict:
        if DRY_RUN:
            log.info("[DRY RUN] Poly order: %s %s %.4f @ %.4f", token_id, side, size, price)
            return {"dry_run": True}
        payload = {
            "token_id": token_id,
            "side":     side,
            "price":    str(price),
            "size":     str(size),
            "type":     "GTC",
        }
        resp = self.session.post(f"{POLY_BASE}/order", json=payload)
        resp.raise_for_status()
        return resp.json()

## 4 · Market Matching

In [ ]:
import re

@dataclass
class MarketPair:
    """A matched market that exists on both platforms."""
    description:       str
    kalshi_ticker:     str
    poly_yes_token_id: str
    poly_no_token_id:  str


def normalize(text: str) -> str:
    """Lower-case, strip punctuation, collapse whitespace for fuzzy matching."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def match_markets(kalshi_client: KalshiClient, poly_client: PolymarketClient) -> list[MarketPair]:
    """
    Fetch all open markets from both platforms and return pairs
    that appear to cover the same event (via keyword overlap).
    """
    # -- Kalshi --
    log.info("Fetching Kalshi markets...")
    kalshi_markets = {}
    cursor = ""
    while True:
        data = kalshi_client.get_markets(limit=200, cursor=cursor)
        for m in data.get("markets", []):
            key = normalize(m.get("title", ""))
            kalshi_markets[key] = m
        cursor = data.get("cursor", "")
        if not cursor:
            break
    log.info("Kalshi: %d open markets", len(kalshi_markets))

    # -- Polymarket --
    log.info("Fetching Polymarket markets...")
    poly_markets = {}
    next_cursor = ""
    while True:
        data = poly_client.get_markets(next_cursor=next_cursor)
        for m in data.get("data", []):
            if not m.get("active"):
                continue
            key = normalize(m.get("question", ""))
            poly_markets[key] = m
        next_cursor = data.get("next_cursor", "")
        if not next_cursor or next_cursor == "LTE=":
            break
    log.info("Polymarket: %d open markets", len(poly_markets))

    # -- Match by normalized title overlap --
    pairs: list[MarketPair] = []
    for k_key, k_mkt in kalshi_markets.items():
        k_words = set(k_key.split())
        for p_key, p_mkt in poly_markets.items():
            p_words = set(p_key.split())
            overlap = k_words & p_words
            # Require at least 4 meaningful words in common
            if len(overlap) >= 4:
                tokens = p_mkt.get("tokens", [])
                yes_token = next((t["token_id"] for t in tokens if t["outcome"] == "Yes"), None)
                no_token  = next((t["token_id"] for t in tokens if t["outcome"] == "No"),  None)
                if yes_token and no_token:
                    pairs.append(MarketPair(
                        description       = k_mkt.get("title", k_key),
                        kalshi_ticker     = k_mkt["ticker"],
                        poly_yes_token_id = yes_token,
                        poly_no_token_id  = no_token,
                    ))
    log.info("Matched %d market pairs", len(pairs))
    return pairs

## 5 · Arbitrage Calculator

In [ ]:
@dataclass
class ArbOpportunity:
    pair:               MarketPair
    direction:          str       # 'YES_POLY_NO_KALSHI' or 'YES_KALSHI_NO_POLY'
    cost:               float     # total cost per $1 payout (0-1)
    ev:                 float     # expected profit per $1 payout
    profit_pct:         float     # ev / cost
    leg_a_price:        float     # first leg ask
    leg_b_price:        float     # second leg ask
    trade_size_usd:     float     # recommended trade size


def net_cost_after_fees(price: float, platform: str) -> float:
    """
    Adjust raw price for fees.
    Kalshi charges 7% of the payout (not the cost), so net cost = price + 0.07*(1-price).
    Polymarket charges ~2% taker on the notional.
    """
    if platform == "kalshi":
        # fee = 7% of payout on the winning leg
        # To be conservative, assume you win: fee = 0.07 * (1 - price)
        return price + KALSHI_FEE_PCT * (1 - price)
    else:
        return price * (1 + POLY_FEE_PCT)


def evaluate_arb(pair: MarketPair, kalshi: KalshiClient, poly: PolymarketClient) -> Optional[ArbOpportunity]:
    """Check one market pair for arbitrage. Returns ArbOpportunity or None."""
    # Raw best-ask prices
    k_yes = kalshi.best_ask(pair.kalshi_ticker, side="yes")
    k_no  = kalshi.best_ask(pair.kalshi_ticker, side="no")
    p_yes = poly.best_ask(pair.poly_yes_token_id)
    p_no  = poly.best_ask(pair.poly_no_token_id)

    if None in (k_yes, k_no, p_yes, p_no):
        return None

    results = []

    # Direction A: buy YES on Poly + buy NO on Kalshi
    cost_a = net_cost_after_fees(p_yes, "poly") + net_cost_after_fees(k_no, "kalshi")
    ev_a   = 1.0 - cost_a
    if ev_a > 0:
        results.append(ArbOpportunity(
            pair           = pair,
            direction      = "YES_POLY_NO_KALSHI",
            cost           = cost_a,
            ev             = ev_a,
            profit_pct     = ev_a / cost_a,
            leg_a_price    = p_yes,
            leg_b_price    = k_no,
            trade_size_usd = min(MAX_TRADE_USD, MAX_TRADE_USD),
        ))

    # Direction B: buy YES on Kalshi + buy NO on Poly
    cost_b = net_cost_after_fees(k_yes, "kalshi") + net_cost_after_fees(p_no, "poly")
    ev_b   = 1.0 - cost_b
    if ev_b > 0:
        results.append(ArbOpportunity(
            pair           = pair,
            direction      = "YES_KALSHI_NO_POLY",
            cost           = cost_b,
            ev             = ev_b,
            profit_pct     = ev_b / cost_b,
            leg_a_price    = k_yes,
            leg_b_price    = p_no,
            trade_size_usd = min(MAX_TRADE_USD, MAX_TRADE_USD),
        ))

    if not results:
        return None

    # Return the best opportunity
    best = max(results, key=lambda x: x.profit_pct)
    return best if best.profit_pct >= MIN_PROFIT_PCT else None

## 6 · Order Executor

In [ ]:
def execute_arb(opp: ArbOpportunity, kalshi: KalshiClient, poly: PolymarketClient) -> None:
    """
    Fire both legs of an arbitrage simultaneously.
    Trade size is scaled so each leg costs at most MAX_TRADE_USD.
    """
    size_usd = opp.trade_size_usd

    log.info(
        "ARB FOUND  %s  |  direction=%s  |  EV=%.4f  |  profit=%.2f%%  |  size=$%.0f",
        opp.pair.description[:60], opp.direction, opp.ev, opp.profit_pct * 100, size_usd,
    )

    if opp.direction == "YES_POLY_NO_KALSHI":
        # Leg A – Polymarket YES
        poly.place_order(
            token_id = opp.pair.poly_yes_token_id,
            side     = "BUY",
            price    = opp.leg_a_price,
            size     = round(size_usd / opp.leg_a_price, 2),
        )
        # Leg B – Kalshi NO
        kalshi.place_order(
            ticker      = opp.pair.kalshi_ticker,
            side        = "no",
            count       = int(size_usd / (opp.leg_b_price)),
            limit_price = int(opp.leg_b_price * 100),
        )

    else:  # YES_KALSHI_NO_POLY
        # Leg A – Kalshi YES
        kalshi.place_order(
            ticker      = opp.pair.kalshi_ticker,
            side        = "yes",
            count       = int(size_usd / (opp.leg_a_price)),
            limit_price = int(opp.leg_a_price * 100),
        )
        # Leg B – Polymarket NO
        poly.place_order(
            token_id = opp.pair.poly_no_token_id,
            side     = "BUY",
            price    = opp.leg_b_price,
            size     = round(size_usd / opp.leg_b_price, 2),
        )

## 7 · Main Scan Loop

In [ ]:
def run_scanner(kalshi: KalshiClient, poly: PolymarketClient, pairs: list[MarketPair]) -> None:
    """
    Continuously scan matched market pairs for arbitrage.
    Re-matches markets every 60 scans (~15 min at 15s interval).
    """
    scan_count = 0
    while True:
        scan_count += 1
        opportunities_found = 0

        for pair in pairs:
            opp = evaluate_arb(pair, kalshi, poly)
            if opp:
                opportunities_found += 1
                execute_arb(opp, kalshi, poly)

        log.info("Scan #%d complete — %d arb(s) found across %d pairs",
                 scan_count, opportunities_found, len(pairs))

        time.sleep(POLL_INTERVAL_S)

## 8 · Run It

In [ ]:
# ── Initialise clients ─────────────────────────────────────────────────────────
kalshi = KalshiClient()
kalshi.login(
    email    = os.environ["KALSHI_EMAIL"],
    password = os.environ["KALSHI_PASSWORD"],
)

poly = PolymarketClient(
    api_key    = os.environ["POLY_API_KEY"],
    api_secret = os.environ["POLY_API_SECRET"],
    passphrase = os.environ["POLY_PASSPHRASE"],
)

# ── Match markets once ─────────────────────────────────────────────────────────
pairs = match_markets(kalshi, poly)
print(f"\nMatched {len(pairs)} market pairs. Sample:")
for p in pairs[:5]:
    print(" •", p.description)

In [ ]:
# ── One-shot scan (safe to run repeatedly in a notebook) ───────────────────────
print(f"DRY_RUN={DRY_RUN}  MIN_PROFIT={MIN_PROFIT_PCT*100:.0f}%  MAX_TRADE=${MAX_TRADE_USD}\n")

for pair in pairs:
    opp = evaluate_arb(pair, kalshi, poly)
    if opp:
        print(f"  OPPORTUNITY  {opp.direction}")
        print(f"    Market   : {opp.pair.description[:70]}")
        print(f"    Leg A    : {opp.leg_a_price:.4f}")
        print(f"    Leg B    : {opp.leg_b_price:.4f}")
        print(f"    Total    : {opp.cost:.4f}  →  EV={opp.ev:.4f}  profit={opp.profit_pct*100:.2f}%")
        execute_arb(opp, kalshi, poly)
        print()

In [ ]:
# ── Continuous scanner (blocks until interrupted) ──────────────────────────────
# Uncomment to run:
# run_scanner(kalshi, poly, pairs)